# Week 7 Maintainer's Copilot — Fine-Tuned Transformer Classifier

This notebook trains the second required classifier for the Week 7 project:

```text
Fine-tuned Transformer: distilbert-base-uncased
```

It uses the already validated Pandas GitHub issues dataset:

```text
issues_processed_pandas_label_fetch.csv
```

Required labels:

- `bug`
- `feature`
- `docs`
- `question`

This notebook saves:

- transformer metrics
- confusion matrix
- model card
- SHA-256 manifest
- trained model folder


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib transformers datasets evaluate accelerate torch tqdm

In [ ]:
import os
import re
import json
import time
import shutil
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)


## Step 1 — Settings

Upload `issues_processed_pandas_label_fetch.csv` to Colab before running this notebook.

This file came from the classical baseline notebook and contains the validated dataset.


In [ ]:
DATA_PATH = Path("issues_processed_pandas_label_fetch.csv")

MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = Path("transformer_model_pandas")
METRICS_PATH = Path("transformer_metrics_pandas.json")
MODEL_CARD_PATH = Path("model_card_pandas_transformer.json")
CONFUSION_MATRIX_PATH = Path("transformer_confusion_matrix_pandas.png")

PROJECT_LABELS = ["bug", "feature", "docs", "question"]

label2id = {label: idx for idx, label in enumerate(PROJECT_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

SEED = 42
set_seed(SEED)

print("Labels:", PROJECT_LABELS)
print("label2id:", label2id)
print("Using GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Step 2 — Load dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Upload issues_processed_pandas_label_fetch.csv to Colab first."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"clean_text", "mapped_label", "split"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df = df.dropna(subset=["clean_text", "mapped_label", "split"]).copy()
df["mapped_label"] = df["mapped_label"].astype(str)

print("Dataset size:", len(df))
display(df["mapped_label"].value_counts())
display(pd.crosstab(df["split"], df["mapped_label"]))
df.head()


## Step 3 — Encode labels

In [ ]:
unknown_labels = set(df["mapped_label"].unique()) - set(PROJECT_LABELS)
if unknown_labels:
    raise ValueError(f"Unknown labels found: {unknown_labels}")

df["label"] = df["mapped_label"].map(label2id)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

display(train_df["mapped_label"].value_counts())
display(val_df["mapped_label"].value_counts())
display(test_df["mapped_label"].value_counts())


## Step 4 — Create Hugging Face datasets

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["clean_text", "label"]].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[["clean_text", "label"]].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[["clean_text", "label"]].reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset = val_dataset.map(tokenize_batch, batched=True)
test_dataset = test_dataset.map(tokenize_batch, batched=True)

train_dataset = train_dataset.remove_columns(["clean_text"])
val_dataset = val_dataset.remove_columns(["clean_text"])
test_dataset = test_dataset.remove_columns(["clean_text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset)


## Step 5 — Load DistilBERT classifier

Freeze policy for first run:

```text
No freezing. The full DistilBERT encoder and classification head are fine-tuned.
```

Why:

```text
The dataset is not huge, but it is balanced enough. Full fine-tuning gives the model a fair chance to adapt to GitHub issue language.
```

Document this later in `DECISIONS.md` and `model_card`.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(PROJECT_LABELS),
    id2label=id2label,
    label2id=label2id,
)

model


## Step 6 — Metrics function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


## Step 7 — Training configuration

The values are intentionally small enough for Colab.

If training is slow, reduce `num_train_epochs` from 3 to 2.


In [ ]:
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=25,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)


## Step 8 — Train

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
train_result


## Step 9 — Evaluate on validation and test

In [ ]:
val_metrics = trainer.evaluate(eval_dataset=val_dataset)
test_metrics = trainer.evaluate(eval_dataset=test_dataset)

print("Validation metrics:")
print(val_metrics)

print("\nTest metrics:")
print(test_metrics)


## Step 10 — Predictions and classification report

In [ ]:
test_output = trainer.predict(test_dataset)
test_logits = test_output.predictions
test_pred_ids = np.argmax(test_logits, axis=-1)

test_true_ids = test_df["label"].to_numpy()

test_true_labels = [id2label[int(x)] for x in test_true_ids]
test_pred_labels = [id2label[int(x)] for x in test_pred_ids]

print(classification_report(
    test_true_labels,
    test_pred_labels,
    labels=PROJECT_LABELS,
    zero_division=0,
))


## Step 11 — Confusion matrix

In [ ]:
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=PROJECT_LABELS)

plt.figure(figsize=(7, 5))
plt.imshow(cm)
plt.title("Fine-Tuned Transformer Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(range(len(PROJECT_LABELS)), PROJECT_LABELS, rotation=45)
plt.yticks(range(len(PROJECT_LABELS)), PROJECT_LABELS)
plt.colorbar()

for i in range(len(PROJECT_LABELS)):
    for j in range(len(PROJECT_LABELS)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150)
plt.show()

print(f"Saved confusion matrix to: {CONFUSION_MATRIX_PATH}")


## Step 12 — Save model and tokenizer

In [ ]:
FINAL_MODEL_DIR = Path("final_transformer_model_pandas")

if FINAL_MODEL_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DIR)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print(f"Saved final model to: {FINAL_MODEL_DIR}")


## Step 13 — Compute SHA-256 for saved model files

This supports the project requirement that model artifacts should have a hash/check later.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


artifact_hashes = {}

for path in sorted(FINAL_MODEL_DIR.rglob("*")):
    if path.is_file():
        artifact_hashes[str(path)] = sha256_file(path)

artifact_hashes


## Step 14 — Save metrics and model card

In [ ]:
final_test_accuracy = accuracy_score(test_true_labels, test_pred_labels)
final_test_macro_f1 = f1_score(test_true_labels, test_pred_labels, average="macro")
final_test_weighted_f1 = f1_score(test_true_labels, test_pred_labels, average="weighted")

per_class_report = classification_report(
    test_true_labels,
    test_pred_labels,
    labels=PROJECT_LABELS,
    output_dict=True,
    zero_division=0,
)

metrics = {
    "repo": "pandas-dev/pandas",
    "model": MODEL_NAME,
    "task": "GitHub issue classification",
    "labels": PROJECT_LABELS,
    "validation_metrics": val_metrics,
    "test_metrics_from_trainer": test_metrics,
    "test_accuracy": float(final_test_accuracy),
    "test_macro_f1": float(final_test_macro_f1),
    "test_weighted_f1": float(final_test_weighted_f1),
    "per_class_report": per_class_report,
    "created_at": datetime.utcnow().isoformat() + "Z",
}

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

model_card = {
    "architecture": MODEL_NAME,
    "task": "issue classification",
    "repo": "pandas-dev/pandas",
    "labels": PROJECT_LABELS,
    "hyperparameters": {
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
    },
    "freeze_policy": "No freezing. Full DistilBERT encoder and classification head were fine-tuned.",
    "training_data": {
        "file": str(DATA_PATH),
        "train_size": int(len(train_df)),
        "validation_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "class_distribution": df["mapped_label"].value_counts().to_dict(),
        "split_distribution": pd.crosstab(df["split"], df["mapped_label"]).to_dict(),
    },
    "final_metrics": {
        "test_accuracy": float(final_test_accuracy),
        "test_macro_f1": float(final_test_macro_f1),
        "test_weighted_f1": float(final_test_weighted_f1),
    },
    "artifact_hashes": artifact_hashes,
    "created_at": datetime.utcnow().isoformat() + "Z",
}

with MODEL_CARD_PATH.open("w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)

print(f"Saved metrics to: {METRICS_PATH}")
print(f"Saved model card to: {MODEL_CARD_PATH}")
model_card


## Step 15 — Zip final model folder for download

Download this zip and keep it as the transformer artifact.


In [ ]:
zip_path = shutil.make_archive("final_transformer_model_pandas", "zip", FINAL_MODEL_DIR)

print("Created zip:", zip_path)
print("Also download:")
print("-", METRICS_PATH)
print("-", MODEL_CARD_PATH)
print("-", CONFUSION_MATRIX_PATH)


# What to send back for review

After running this notebook, send:

```text
transformer_metrics_pandas.json
model_card_pandas_transformer.json
transformer_confusion_matrix_pandas.png
```

Optional if needed:

```text
final_transformer_model_pandas.zip
```
